In [ ]:

from getpass import getpass
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_histogram

# STEP 1: Connect to IBM Quantum
api_key = getpass("Enter your IBM Quantum API key: ")
instance = input("Enter your IBM Quantum instance / CRN: ").strip()

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=api_key,
    instance=instance
)

print("Connected to IBM Quantum successfully!")

# STEP 2: Select an available real quantum backend
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

print("Selected backend:", backend.name)

# STEP 3: Original Bell-state circuit
# H creates superposition; CX creates correlation
qc_original = QuantumCircuit(2)
qc_original.h(0)
qc_original.cx(0, 1)
qc_original.measure_all()

# STEP 4: Modified circuit
# Add one X gate to qubit 1 to reverse its measurement
qc_modified = QuantumCircuit(2)
qc_modified.h(0)
qc_modified.cx(0, 1)
qc_modified.x(1)
qc_modified.measure_all()

# STEP 5: Display both circuits
print("\nOriginal Bell-state circuit:")
display(qc_original.draw("mpl"))

print("\nModified circuit with one X gate:")
display(qc_modified.draw("mpl"))

# STEP 6: Predictions before execution
print("\nPREDICTIONS")
print("Original: 00 and 11 should dominate.")
print("Modified: 01 and 10 should dominate.")

# STEP 7: Transpile circuits for the selected backend
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1
)

isa_circuits = pm.run([qc_original, qc_modified])

# STEP 8: Run each circuit for 100 shots
sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=100)

print("\nJob ID:", job.job_id())
result = job.result()

# STEP 9: Retrieve measurement counts
counts_original = result[0].data.meas.get_counts()
counts_modified = result[1].data.meas.get_counts()

print("\nORIGINAL CIRCUIT RESULTS:")
print(counts_original)

print("\nMODIFIED CIRCUIT RESULTS:")
print(counts_modified)

# STEP 10: Plot histograms
plot_histogram(
    [counts_original, counts_modified],
    legend=["Original Bell Circuit", "Modified Circuit"],
    title="Challenge 3: Changing Quantum Correlation"
)
plt.show()

# STEP 11: Compare the results
print("\nCOMPARISON")
print("Original circuit: correlated measurement outcomes.")
print("Modified circuit: opposite measurement outcomes.")
print("The X gate flips qubit 1, changing 00/11 into 01/10.")
print("Real hardware may show small deviations due to noise.")


Enter your IBM Quantum API key:  ········
